In [1]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "sincere-quasar-511010-v3"
client = bigquery.Client(project=PROJECT_ID)

query = """
WITH week_dates AS (
    SELECT
        wm_yr_wk AS week_id,
        MIN(DATE(date)) AS ds
    FROM `sincere-quasar-511010-v3.retail_demand_raw.calendar`
    GROUP BY wm_yr_wk
),
top_series AS (
    SELECT item_id, store_id
    FROM `sincere-quasar-511010-v3.retail_demand_dbt.fct_weekly_sales`
    GROUP BY item_id, store_id
    ORDER BY SUM(total_units_sold) DESC
    LIMIT 10
)
SELECT
    f.item_id,
    f.store_id,
    d.ds,
    f.total_units_sold AS y
FROM `sincere-quasar-511010-v3.retail_demand_dbt.fct_weekly_sales` f
JOIN week_dates d USING (week_id)
JOIN top_series t USING (item_id, store_id)
ORDER BY item_id, store_id, ds
"""

df = client.query(query).to_dataframe()
df["ds"] = pd.to_datetime(df["ds"])

print("Loaded successfully:", df.shape)
display(df.head())

c:\Users\babyp\Project-3-Retail-Demand-Forecasting-Inventory-Optimization\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded successfully: (2740, 4)


,item_id,store_id,ds,y
0,FOODS_3_090,CA_1,2011-01-29,616
1,FOODS_3_090,CA_1,2011-02-05,323
2,FOODS_3_090,CA_1,2011-02-12,0
3,FOODS_3_090,CA_1,2011-02-19,0
4,FOODS_3_090,CA_1,2011-02-26,0


In [2]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = df.sort_values(["item_id", "store_id", "ds"]).copy()

group = df.groupby(["item_id", "store_id"])["y"]

df["lag_1"] = group.shift(1)
df["lag_2"] = group.shift(2)
df["lag_4"] = group.shift(4)
df["rolling_mean_4"] = group.transform(
    lambda s: s.shift(1).rolling(4).mean()
)
df["month"] = df["ds"].dt.month
df["week_of_year"] = df["ds"].dt.isocalendar().week.astype(int)

df = df.dropna().copy()

In [3]:
feature_cols = [
    "lag_1", "lag_2", "lag_4",
    "rolling_mean_4", "month", "week_of_year"
]

cutoff = df["ds"].max() - pd.Timedelta(weeks=8)
train = df[df["ds"] <= cutoff]
test = df[df["ds"] > cutoff]

model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

model.fit(train[feature_cols], train["y"])
test = test.copy()
test["prediction"] = np.maximum(
    0, model.predict(test[feature_cols])
)

mae = mean_absolute_error(test["y"], test["prediction"])
rmse = np.sqrt(mean_squared_error(test["y"], test["prediction"]))

print(f"LightGBM MAE: {mae:.2f}")
print(f"LightGBM RMSE: {rmse:.2f}")

LightGBM MAE: 76.20
LightGBM RMSE: 120.67


In [4]:
print("Prediction-related variables:")
print([
    name for name in globals()
    if any(word in name.lower()
           for word in ["pred", "forecast", "actual", "y_test", "y_pred"])
])

print("\nDataFrames containing item/store information:")
for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        if {"item_id", "store_id"}.issubset(obj.columns):
            print(f"\n{name}: shape={obj.shape}")
            print("Columns:", obj.columns.tolist())
            display(obj.head(3))

Prediction-related variables:
[]

DataFrames containing item/store information:

df: shape=(2700, 10)
Columns: ['item_id', 'store_id', 'ds', 'y', 'lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'month', 'week_of_year']


,item_id,store_id,ds,y,lag_1,lag_2,lag_4,rolling_mean_4,month,week_of_year
4,FOODS_3_090,CA_1,2011-02-26,0,0,0,616,234.75,2,8
5,FOODS_3_090,CA_1,2011-03-05,0,0,0,323,80.75,3,9
6,FOODS_3_090,CA_1,2011-03-12,0,0,0,0,0.00,3,10



train: shape=(2620, 10)
Columns: ['item_id', 'store_id', 'ds', 'y', 'lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'month', 'week_of_year']


,item_id,store_id,ds,y,lag_1,lag_2,lag_4,rolling_mean_4,month,week_of_year
4,FOODS_3_090,CA_1,2011-02-26,0,0,0,616,234.75,2,8
5,FOODS_3_090,CA_1,2011-03-05,0,0,0,323,80.75,3,9
6,FOODS_3_090,CA_1,2011-03-12,0,0,0,0,0.00,3,10



test: shape=(80, 11)
Columns: ['item_id', 'store_id', 'ds', 'y', 'lag_1', 'lag_2', 'lag_4', 'rolling_mean_4', 'month', 'week_of_year', 'prediction']


,item_id,store_id,ds,y,lag_1,lag_2,lag_4,rolling_mean_4,month,week_of_year,prediction
266,FOODS_3_090,CA_1,2016-03-05,367,402,380,321,356.25,3,9,396.905569
267,FOODS_3_090,CA_1,2016-03-12,312,367,402,322,367.75,3,10,365.029171
268,FOODS_3_090,CA_1,2016-03-19,371,312,367,380,365.25,3,11,343.080742


In [5]:
# Prepare LightGBM predictions for BigQuery
import pandas as pd
import numpy as np

forecast_upload = test[
    ["item_id", "store_id", "ds", "y", "prediction"]
].copy()

forecast_upload = forecast_upload.rename(columns={
    "ds": "forecast_date",
    "y": "actual_units",
    "prediction": "predicted_units"
})

forecast_upload["forecast_date"] = pd.to_datetime(
    forecast_upload["forecast_date"]
).dt.date

forecast_upload["model_name"] = "LightGBM"
forecast_upload["mae"] = 76.20

# Keep the exact column order required by BigQuery
forecast_upload = forecast_upload[
    [
        "item_id",
        "store_id",
        "forecast_date",
        "model_name",
        "actual_units",
        "predicted_units",
        "mae"
    ]
]

print("Prepared rows:", len(forecast_upload))
print("Columns:", forecast_upload.columns.tolist())
display(forecast_upload.head())

Prepared rows: 80
Columns: ['item_id', 'store_id', 'forecast_date', 'model_name', 'actual_units', 'predicted_units', 'mae']


,item_id,store_id,forecast_date,model_name,actual_units,predicted_units,mae
266,FOODS_3_090,CA_1,2016-03-05,LightGBM,367,396.905569,76.2
267,FOODS_3_090,CA_1,2016-03-12,LightGBM,312,365.029171,76.2
268,FOODS_3_090,CA_1,2016-03-19,LightGBM,371,343.080742,76.2
269,FOODS_3_090,CA_1,2016-03-26,LightGBM,402,359.992031,76.2
270,FOODS_3_090,CA_1,2016-04-02,LightGBM,317,400.317435,76.2


In [6]:
print("Missing values:")
print(forecast_upload.isnull().sum())

print("\nDuplicate item-store-date rows:")
print(
    forecast_upload.duplicated(
        subset=["item_id", "store_id", "forecast_date", "model_name"]
    ).sum()
)

print("\nData types:")
print(forecast_upload.dtypes)

print("\nPrediction summary:")
print(forecast_upload["predicted_units"].describe())

Missing values:
item_id            0
store_id           0
forecast_date      0
model_name         0
actual_units       0
predicted_units    0
mae                0
dtype: int64

Duplicate item-store-date rows:
0

Data types:
item_id                str
store_id               str
forecast_date       object
model_name             str
actual_units         Int64
predicted_units    float64
mae                float64
dtype: object

Prediction summary:
count     80.000000
mean     482.572220
std      114.406881
min      330.915335
25%      406.243359
50%      450.206866
75%      551.838662
max      824.910165
Name: predicted_units, dtype: float64


In [7]:
from google.cloud import bigquery

PROJECT_ID = "sincere-quasar-511010-v3"
TABLE_ID = f"{PROJECT_ID}.retail_demand_raw.forecast_results"

client = bigquery.Client(project=PROJECT_ID)

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    schema=[
        bigquery.SchemaField("item_id", "STRING"),
        bigquery.SchemaField("store_id", "STRING"),
        bigquery.SchemaField("forecast_date", "DATE"),
        bigquery.SchemaField("model_name", "STRING"),
        bigquery.SchemaField("actual_units", "FLOAT64"),
        bigquery.SchemaField("predicted_units", "FLOAT64"),
        bigquery.SchemaField("mae", "FLOAT64"),
    ],
)

job = client.load_table_from_dataframe(
    forecast_upload,
    TABLE_ID,
    job_config=job_config,
)

job.result()

print("Upload completed successfully!")
print("Rows uploaded:", job.output_rows)
print("Destination:", TABLE_ID)

Upload completed successfully!
Rows uploaded: 80
Destination: sincere-quasar-511010-v3.retail_demand_raw.forecast_results


### BigQuery Forecast Storage

- Stored 80 LightGBM test predictions in BigQuery.
- **Destination:** `sincere-quasar-511010-v3.retail_demand_raw.forecast_results`
- **Fields stored:** Item ID, store ID, forecast date, model name, actual units, predicted units, and MAE.
- **Validation:** No missing actual or predicted values.
- **LightGBM evaluation metrics:** MAE = 76.20 and RMSE = 120.67.